# La familia Claude 4.X: Opus 4.7, Sonnet 4.6 y Haiku 4.5

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/claude-4/01-familia-claude-4.ipynb)

En este notebook exploraremos las diferencias prácticas entre los tres modelos de la familia Claude 4.X y cómo elegir el más adecuado para cada tarea.

In [ ]:
!pip install anthropic -q

In [ ]:
import anthropic
import os
import time

# Configura tu API key
# os.environ['ANTHROPIC_API_KEY'] = 'tu-api-key'
client = anthropic.Anthropic()

MODELOS = {
    'opus': 'claude-opus-4-7',
    'sonnet': 'claude-sonnet-4-6',
    'haiku': 'claude-haiku-4-5-20251001',
}
print('Modelos disponibles:', MODELOS)

## 1. Comparativa de velocidad y coste

Vamos a enviar la misma pregunta a los tres modelos y medir latencia y uso de tokens.

In [ ]:
def medir_modelo(modelo_key: str, prompt: str) -> dict:
    inicio = time.time()
    response = client.messages.create(
        model=MODELOS[modelo_key],
        max_tokens=256,
        messages=[{'role': 'user', 'content': prompt}],
    )
    latencia = time.time() - inicio
    return {
        'modelo': modelo_key,
        'latencia_s': round(latencia, 2),
        'tokens_input': response.usage.input_tokens,
        'tokens_output': response.usage.output_tokens,
        'respuesta': response.content[0].text[:200],
    }

prompt_test = 'Explica en 3 frases qué es el aprendizaje automático.'

resultados = []
for key in ['haiku', 'sonnet']:  # Omitimos opus para ahorrar coste en el demo
    r = medir_modelo(key, prompt_test)
    resultados.append(r)
    print(f"\n{'='*50}")
    print(f"Modelo: {r['modelo']}")
    print(f"Latencia: {r['latencia_s']}s")
    print(f"Tokens: {r['tokens_input']} input / {r['tokens_output']} output")
    print(f"Respuesta: {r['respuesta']}")

## 2. Router de modelos automático

Implementamos un router que selecciona el modelo más económico capaz de resolver la tarea.

In [ ]:
import re

PATRONES_COMPLEJOS = re.compile(
    r'\b(analiza|diseña|compara|razona|optimiza|arquitectura|estrategia|exhaustivo)\b',
    re.IGNORECASE
)
PATRONES_SIMPLES = re.compile(
    r'\b(clasifica|sí o no|verdadero o falso|traduce brevemente|qué significa)\b',
    re.IGNORECASE
)

def router(prompt: str) -> tuple[str, str]:
    palabras = len(prompt.split())
    if PATRONES_COMPLEJOS.search(prompt) or palabras > 200:
        modelo_key = 'opus'
    elif PATRONES_SIMPLES.search(prompt) or palabras < 25:
        modelo_key = 'haiku'
    else:
        modelo_key = 'sonnet'
    
    response = client.messages.create(
        model=MODELOS[modelo_key],
        max_tokens=512,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return modelo_key, response.content[0].text

casos = [
    'Traduce brevemente: hello world',
    'Explica qué es una red neuronal y cómo aprende.',
    'Diseña una arquitectura de microservicios para un SaaS con 100K usuarios concurrentes.',
]

for caso in casos:
    modelo_usado, respuesta = router(caso)
    print(f"\nPrompt: {caso[:60]}...")
    print(f"Modelo seleccionado: {modelo_usado.upper()}")
    print(f"Respuesta: {respuesta[:150]}...")

## 3. Tabla comparativa de precios

Visualizamos el coste estimado por millón de tokens para cada modelo.

In [ ]:
precios = {
    'Claude Opus 4.7':  {'input': 15.0,  'output': 75.0},
    'Claude Sonnet 4.6': {'input': 3.0,  'output': 15.0},
    'Claude Haiku 4.5':  {'input': 0.80, 'output': 4.0},
}

print(f"{'Modelo':<22} {'Input $/M':<12} {'Output $/M':<12} {'Ratio vs Opus'}")
print('-' * 60)
for modelo, p in precios.items():
    ratio = precios['Claude Opus 4.7']['output'] / p['output']
    print(f"{modelo:<22} ${p['input']:<11.2f} ${p['output']:<11.2f} {ratio:.0f}x más barato")

## Conclusión

- **Haiku 4.5**: tareas simples, alto volumen, baja latencia
- **Sonnet 4.6**: producción general, mejor relación calidad/coste
- **Opus 4.7**: razonamiento complejo, Extended Thinking, decisiones críticas

Un router de modelos puede reducir el coste total un **60-80%** sin pérdida de calidad percibida.